# ArchiGen 3D - training on Kaggle (real data, ~1 hour)

**Before running:** right panel → *Session options* → **Accelerator = GPU T4 x2**, **Internet = On**
(internet is only needed for `pip install lpips` and the LPIPS weights).
Add your uploaded dataset with **+ Add Input**.

The notebook adapts to what you attached (for floorplans, either `archigen_cubicasa.zip` **or** the public *CubiCasa5k* dataset, which is rasterized here automatically):
- **archigen_facades.zip only** → L1 baseline on GPU 0 and Pix2Pix on GPU 1, ~50 min each (full protocol, 256×256).
- **+ the CubiCasa dataset** → GPU 0: L1 then Pix2Pix (~18 + ~32 min); GPU 1: UNet++ at **256×256 + mixed precision** (*reduced budget*, documented).

Each training stops *before* an epoch that would exceed its budget; the number of completed
epochs is written to `models/metrics/*_val.json` (`epochs_completed`, `stopped_by_time_budget`).
Model selection uses **validation only**; the test split is evaluated once at the end.

In [ ]:
# 1) Setup: locate the uploaded bundle, build a writable workspace, check GPUs
import glob, os, shutil, subprocess, sys, time
from pathlib import Path

def find(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return Path(hits[0]) if hits else None

code = find('/kaggle/input/**/archigen/params.yaml')
FACADES = find('/kaggle/input/**/archigen/data/raw/facades')
CUBICASA = find('/kaggle/input/**/archigen/data/processed/cubicasa_cache')
assert code, "Code not found: '+ Add Input' -> add the dataset made from archigen_facades.zip (or archigen_kaggle.zip)"
SRC = code.parent
HAS_FACADES, HAS_FP = FACADES is not None, CUBICASA is not None
print('facades attached:', HAS_FACADES, '| cubicasa attached:', HAS_FP)

W = Path('/kaggle/working/archigen')
if W.exists():
    shutil.rmtree(W)
(W / 'data/raw').mkdir(parents=True)
(W / 'data/processed').mkdir(parents=True)
shutil.copytree(SRC / 'src', W / 'src')
shutil.copy(SRC / 'params.yaml', W / 'params.yaml')
# Data stays read-only in /kaggle/input (no copy): symlinks only
if HAS_FACADES:
    os.symlink(FACADES, W / 'data/raw/facades')
if HAS_FP:
    os.symlink(CUBICASA, W / 'data/processed/cubicasa_cache')
else:
    # No pre-rasterized cache: use the PUBLIC CubiCasa5K dataset (raw SVG + PNG) if attached,
    # and rasterize it here with the official splits (a few minutes on Kaggle CPUs).
    raw = [Path(t).parent for t in glob.glob('/kaggle/input/**/train.txt', recursive=True)
           if (Path(t).parent / 'val.txt').exists() and (Path(t).parent / 'test.txt').exists()]
    if raw:
        print('Raw CubiCasa5K found at', raw[0], '-> rasterizing (official splits)...')
        r = subprocess.run([sys.executable, '-m', 'src.data.rasterize_cubicasa', '--source', str(raw[0]),
                            '--out', str(W / 'data/processed/cubicasa_cache'), '--workers', str(os.cpu_count() or 4)],
                           cwd=W, capture_output=True, text=True)
        print(r.stdout[-600:], r.stderr[-600:])
        HAS_FP = r.returncode == 0 and any((W / 'data/processed/cubicasa_cache/train').glob('*.npz'))
        print('cubicasa ready:', HAS_FP)
os.chdir(W)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lpips'])

import torch
N_GPU = torch.cuda.device_count()
print('torch', torch.__version__, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert N_GPU > 0, 'No GPU: Session options > Accelerator > GPU T4 x2'
for split in ('train', 'val', 'test'):
    print(split, 'facades:', len(list((W / 'data/raw/facades' / split).glob('*.jpg'))) if HAS_FACADES else '-',
          '| cubicasa:', len(list((W / 'data/processed/cubicasa_cache' / split).glob('*.npz'))) if HAS_FP else '-')

In [ ]:
# 2) Train everything in parallel (blocks until finished, prints progress every minute)
TOTAL_MIN = 55   # overall wall-clock target

PY = sys.executable
L1  = f'{PY} -m src.training.train_facade --mode l1_baseline --workers 2'
P2P = f'{PY} -m src.training.train_facade --mode pix2pix --workers 2'
FP  = f'{PY} -m src.training.train_floorplan --image-size 256 --amp --workers 3'

two = N_GPU >= 2
if HAS_FACADES and HAS_FP and two:      # GPU0: both facade models, GPU1: floorplan
    lanes = {'facades':   (0, f'{L1} --time-budget-min 18 ; {P2P} --time-budget-min 32'),
             'floorplan': (1, f'{FP} --time-budget-min {TOTAL_MIN - 5}')}
elif HAS_FACADES and HAS_FP:           # one GPU: everything in sequence
    lanes = {'all': (0, f'{L1} --time-budget-min 10 ; {P2P} --time-budget-min 18 ; '
                        f'{FP} --time-budget-min {TOTAL_MIN - 33}')}
elif HAS_FACADES and two:              # facades only: one model per GPU, full budget each
    lanes = {'l1_baseline': (0, f'{L1} --time-budget-min {TOTAL_MIN - 5}'),
             'pix2pix':     (1, f'{P2P} --time-budget-min {TOTAL_MIN - 5}')}
elif HAS_FACADES:
    lanes = {'facades': (0, f'{L1} --time-budget-min 20 ; {P2P} --time-budget-min {TOTAL_MIN - 25}')}
else:                                  # floorplan only
    lanes = {'floorplan': (0, f'{FP} --time-budget-min {TOTAL_MIN - 5}')}

procs = {}
for name, (gpu, cmd) in lanes.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1',
               MLFLOW_TRACKING_URI=f'sqlite:///{W}/mlflow_{name}.db')   # one DB per process
    procs[name] = subprocess.Popen(cmd, shell=True, cwd=W, env=env,
                                   stdout=open(W / f'log_{name}.txt', 'w'), stderr=subprocess.STDOUT)
    print(f'started {name} on GPU {gpu}: {cmd}')

t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(60)
    print(f'--- {(time.time() - t0) / 60:.0f} min ---')
    for name in procs:
        lines = (W / f'log_{name}.txt').read_text(errors='ignore').splitlines()
        keep = [l for l in lines if any(k in l for k in ('Epoch', 'budget', 'Error', 'mode=', 'Floorplan]'))]
        print(f'  {name}: {keep[-1] if keep else (lines[-1] if lines else "starting...")}')
print('exit codes:', {n: p.returncode for n, p in procs.items()}, f'| {(time.time() - t0) / 60:.1f} min')
for name in procs:
    if procs[name].returncode != 0:
        print(f'\n===== last lines of log_{name}.txt =====')
        print('\n'.join((W / f'log_{name}.txt').read_text(errors='ignore').splitlines()[-30:]))

In [ ]:
# 3) Final evaluation on the held-out TEST split (once, after training)
jobs = [('facade', 'facade_unet_l1_best.pt'), ('facade', 'facade_pix2pix_best.pt'),
        ('floorplan', 'floorplan_unetpp_best.pt')]
for task, ckpt in jobs:
    p = W / 'models/checkpoints' / ckpt
    print(f'\n===== {ckpt} =====')
    if p.exists():
        subprocess.run([sys.executable, '-m', 'src.evaluation.evaluate', task, '--checkpoint', str(p)], cwd=W)
    else:
        print('MISSING - check the training log above')

In [ ]:
# 4) Validation vs baseline (Cadrage objective 3: Pix2Pix val L1 >= 5% better than the U-Net L1 baseline)
import json
m = lambda n: json.loads((W / 'models/metrics' / n).read_text())
try:
    b, g = m('facade_unet_l1_val.json'), m('facade_pix2pix_val.json')
    lb, lg = b['best']['val_l1'], g['best']['val_l1']
    gain = 100 * (lb - lg) / lb
    print(f"val L1 baseline={lb:.4f} ({b['epochs_completed']} ep) | pix2pix={lg:.4f} ({g['epochs_completed']} ep)")
    print(f"gain = {gain:+.1f} %  ->  target >= 5 %: {'MET' if gain >= 5 else 'NOT MET (report it)'}")
except FileNotFoundError as e:
    print('missing metrics:', e)

In [ ]:
# 5) Package results -> /kaggle/working/archigen_results.zip (download from the Output tab)
R = Path('/kaggle/working/archigen_results')
if R.exists():
    shutil.rmtree(R)
(R / 'checkpoints').mkdir(parents=True)
for f in (W / 'models/checkpoints').glob('*_best.pt'):
    shutil.copy(f, R / 'checkpoints')
shutil.copytree(W / 'models/metrics', R / 'metrics')
if (W / 'models/samples').exists():
    shutil.copytree(W / 'models/samples', R / 'samples')
for f in W.glob('log_*.txt'):
    shutil.copy(f, R)
for f in W.glob('mlflow_*.db'):
    shutil.copy(f, R)
shutil.make_archive('/kaggle/working/archigen_results', 'zip', R)
shutil.rmtree(R)
shutil.rmtree(W)   # remove workspace (symlinks + large *_last.pt) so only the zip is kept as output
print('Ready:', '/kaggle/working/archigen_results.zip',
      f"{Path('/kaggle/working/archigen_results.zip').stat().st_size / 1e6:.0f} MB")